# Laboratório: a mesma tarefa em 3 gerações de NLP

Na parte teórica vimos que NLP existe desde os anos 1950 e que  cada era resolveu **as mesmas tarefas** de um jeito diferente. Agora vamos provar isso com código: **um único problema**,
resolvido três vezes.

**O problema:** triagem automática de mensagens que chegam à coordenação do curso, em 5 categorias, `matricula`, `estagio`, `colacao`, `recurso`, `outro`.

| Geração | Era | Como resolve | Exemplos rotulados |
|---|---|---|---|
| 1 | Estatística (1990–2013) | TF-IDF + Regressão Logística | 100 |
| 2 | Neural pré-Transformer (2013–2017) | Embeddings de sentença + Regressão Logística | 100 |
| 3 | Transformer (2017 →) | LLM em *zero-shot* (só um prompt) | **0** |

### Roteiro (50 min)

| Bloco | Tempo |
|---|---|
| 0. Ambiente e dados | 5 min |
| 1. Geração 1 — TF-IDF | 12 min |
| 2. Geração 2 — Embeddings | 12 min |
| 3. Geração 3 — LLM zero-shot | 15 min |
| 4. Comparação e discussão | 6 min |

## 0. Ambiente e dados

O Colab já vem com `scikit-learn` e `pandas`. Falta o resto:

In [1]:
!pip install -q sentence-transformers huggingface_hub google-genai

In [2]:
import time, json, re, unicodedata
import pandas as pd

RESULTADOS = {}   # cada geração guarda o resultado aqui; a tabela final se monta sozinha

### O dataset

**100 mensagens de treino** (20 por categoria) e **dois conjuntos de teste com as mesmas 25 intenções**, escritos de dois jeitos:

- `TESTE_FORMAL`: português correto, usando a forma culta da linguagem.
- `TESTE_REAL`: do jeito que escreve no dia a dia: sem acento, com abreviação e typo
  (*"acabei todas as materia, quando eu pego o canudo?"*).

Esse par não é enfeite: é o experimento controlado da aula. **Mesma intenção, escrita diferente.** É exatamente a "discreticidade" e o "fator humano".

Repare também que o vocabulário do teste é **de propósito** diferente do treino: quem escreve "canudo" nunca escreveu "colação de grau".

> ⚠️ **Um aviso que vale para o resto da aula:** 25 exemplos de teste é **pouco**. Cada erro mexe 4 pontos percentuais na acurácia. Diferenças menores que ~10 pontos entre as gerações, aqui, não significam nada, são ruído. Vamos usar esses números para *conversar*, não para eleger um vencedor.

In [3]:
# Dataset de triagem de solicitações acadêmicas (PT-BR)
# 100 mensagens de treino + dois conjuntos de teste com as MESMAS 25 intenções:
#   TESTE_FORMAL — português correto     |  TESTE_REAL — como escreve no dia a dia
TREINO = [
    ("Não consegui fazer a matrícula no portal, dá erro de senha.", "matricula"),
    ("Quero me matricular na disciplina de Cálculo II neste semestre.", "matricula"),
    ("Perdi o prazo de matrícula, ainda dá tempo de fazer?", "matricula"),
    ("Preciso trancar minha matrícula por motivo de saúde.", "matricula"),
    ("Como faço para incluir uma disciplina optativa na minha matrícula?", "matricula"),
    ("O sistema não deixa eu me matricular porque diz que tenho pendência.", "matricula"),
    ("Gostaria de saber quando abre o período de matrícula do próximo semestre.", "matricula"),
    ("Quero cancelar a matrícula em Banco de Dados II.", "matricula"),
    ("Fiz a matrícula mas a disciplina não apareceu no meu horário.", "matricula"),
    ("Sou aluno novo e não sei como realizar minha primeira matrícula.", "matricula"),
    ("Preciso de ajuda para me rematricular, fiquei um semestre parado.", "matricula"),
    ("A matrícula em Estrutura de Dados está com as vagas esgotadas, tem como abrir mais uma?", "matricula"),
    ("Quero mudar de turno, da manhã para a noite, na minha matrícula.", "matricula"),
    ("Meu nome não consta na lista de matriculados da turma de Redes.", "matricula"),
    ("Como faço para me matricular em uma disciplina de outro curso?", "matricula"),
    ("Solicito reabertura de matrícula, estava afastado.", "matricula"),
    ("Vou trancar o semestre, quais documentos preciso entregar?", "matricula"),
    ("Fiz a inscrição nas disciplinas mas não recebi confirmação nenhuma.", "matricula"),
    ("Estou com dúvida sobre quantos créditos posso me matricular por semestre.", "matricula"),
    ("Quero desistir de uma cadeira em que me matriculei por engano.", "matricula"),

    ("Consegui uma vaga de estágio numa empresa, como formalizo?", "estagio"),
    ("Quais documentos preciso para assinar o termo de compromisso de estágio?", "estagio"),
    ("Meu supervisor precisa assinar o relatório de estágio, para quem envio?", "estagio"),
    ("Posso contar o trabalho que já tenho como estágio obrigatório?", "estagio"),
    ("Quantas horas de estágio preciso cumprir para me formar?", "estagio"),
    ("A empresa quer renovar meu estágio por mais seis meses.", "estagio"),
    ("Quem é o professor orientador de estágio do curso?", "estagio"),
    ("Entreguei o relatório final de estágio mas ainda não vi a nota.", "estagio"),
    ("Estou procurando vaga de estágio, a coordenação divulga oportunidades?", "estagio"),
    ("Preciso do convênio da instituição com a empresa para começar a estagiar.", "estagio"),
    ("Posso fazer estágio não obrigatório no terceiro período?", "estagio"),
    ("Meu estágio terminou antes do prazo, o que faço com a documentação?", "estagio"),
    ("O termo aditivo do meu estágio está parado há duas semanas.", "estagio"),
    ("Como comprovo as horas do estágio supervisionado?", "estagio"),
    ("A empresa pediu a carta de apresentação da instituição para o estágio.", "estagio"),
    ("Fui efetivado na empresa onde estagiava, isso muda alguma coisa?", "estagio"),
    ("Quero trocar de empresa no meio do estágio obrigatório.", "estagio"),
    ("Qual o modelo de plano de atividades do estágio?", "estagio"),
    ("Estagio em uma startup sem convênio, é possível regularizar?", "estagio"),
    ("Meu relatório de estágio foi reprovado, posso refazer?", "estagio"),

    ("Quando será a cerimônia de colação de grau deste semestre?", "colacao"),
    ("Já integralizei todas as disciplinas, posso colar grau?", "colacao"),
    ("Quero colar grau em gabinete, como solicito?", "colacao"),
    ("Quantos convites tenho direito para a colação?", "colacao"),
    ("Preciso do meu diploma com urgência para um concurso.", "colacao"),
    ("A lista de formandos já foi divulgada?", "colacao"),
    ("Faltou uma disciplina para eu me formar, ainda participo da colação?", "colacao"),
    ("Qual o traje exigido na cerimônia de formatura?", "colacao"),
    ("Solicito antecipação de colação de grau por aprovação em concurso público.", "colacao"),
    ("Onde retiro o certificado de conclusão de curso?", "colacao"),
    ("Meu nome saiu errado na lista de colação de grau.", "colacao"),
    ("Quem é o paraninfo da turma deste ano?", "colacao"),
    ("Posso colar grau mesmo devendo horas complementares?", "colacao"),
    ("Já colei grau, quanto tempo demora para sair o diploma?", "colacao"),
    ("Preciso de declaração de conclusão enquanto o diploma não sai.", "colacao"),
    ("A cerimônia de formatura será presencial ou online?", "colacao"),
    ("Como faço o requerimento de colação de grau antecipada?", "colacao"),
    ("Não poderei comparecer à colação, posso enviar um procurador?", "colacao"),
    ("Quais são os requisitos para participar da solenidade de formatura?", "colacao"),
    ("O ensaio da colação de grau é obrigatório?", "colacao"),

    ("Quero recorrer da nota da segunda avaliação de Algoritmos.", "recurso"),
    ("Discordo do resultado da prova, gostaria de solicitar revisão.", "recurso"),
    ("Fui reprovado por falta mas estava com atestado médico.", "recurso"),
    ("Solicito revisão da minha média final na disciplina de Física.", "recurso"),
    ("O professor não lançou minha nota do trabalho, como contesto?", "recurso"),
    ("Quero abrir recurso contra o indeferimento do meu pedido de auxílio.", "recurso"),
    ("Perdi a prova por motivo de doença, posso fazer segunda chamada?", "recurso"),
    ("Minha frequência está errada no sistema, quero contestar.", "recurso"),
    ("Entrei com recurso semana passada e não tive resposta.", "recurso"),
    ("Como faço para pedir revisão de prova depois do prazo?", "recurso"),
    ("A nota lançada não bate com a que o professor mostrou em sala.", "recurso"),
    ("Solicito reconsideração da decisão da coordenação sobre meu aproveitamento.", "recurso"),
    ("Fui desclassificado do edital de monitoria e quero recorrer.", "recurso"),
    ("Quero contestar a reprovação em Cálculo I, acredito que houve erro de soma.", "recurso"),
    ("Existe prazo para recurso contra resultado de processo seletivo interno?", "recurso"),
    ("Meu pedido de dispensa de disciplina foi negado, posso recorrer?", "recurso"),
    ("Preciso de um formulário para interpor recurso administrativo.", "recurso"),
    ("A banca não considerou minha justificativa, quero recorrer da decisão.", "recurso"),
    ("Solicito reavaliação do meu trabalho final, a correção parece inconsistente.", "recurso"),
    ("Faltei à prova porque estava trabalhando no horário, cabe recurso?", "recurso"),

    ("Bom dia, qual o horário de funcionamento da secretaria?", "outro"),
    ("Preciso de uma declaração de vínculo para o passe estudantil.", "outro"),
    ("O wifi do bloco C está fora do ar.", "outro"),
    ("Como faço para pegar livros emprestados na biblioteca?", "outro"),
    ("Gostaria de saber se haverá aula na próxima segunda, é feriado?", "outro"),
    ("Perdi meu crachá de identificação estudantil.", "outro"),
    ("Onde encontro o calendário acadêmico atualizado?", "outro"),
    ("Tem algum edital de monitoria aberto?", "outro"),
    ("Quero saber sobre o auxílio alimentação do estudante.", "outro"),
    ("O ar-condicionado da sala 302 não está funcionando.", "outro"),
    ("Como participo do grupo de pesquisa do professor?", "outro"),
    ("Existe algum curso de extensão em programação neste semestre?", "outro"),
    ("Meu e-mail institucional parou de funcionar.", "outro"),
    ("Gostaria de agendar uma conversa com a coordenação.", "outro"),
    ("Quais são as regras para uso do laboratório de informática?", "outro"),
    ("Onde vejo as notas do semestre passado?", "outro"),
    ("Preciso atualizar meu endereço no cadastro.", "outro"),
    ("Tem estacionamento para alunos no campus?", "outro"),
    ("Quero saber se o restaurante universitário funciona no sábado.", "outro"),
    ("A impressora da biblioteca está sem papel.", "outro"),
]

TESTE_FORMAL = [
    ("Não consegui garantir vaga nas cadeiras que escolhi para o próximo período.", "matricula"),
    ("Como faço pra sair de uma disciplina que peguei sem querer?", "matricula"),
    ("Quero suspender meus estudos por um semestre e voltar depois.", "matricula"),
    ("O portal não aceita minha inscrição nas turmas, aparece um erro.", "matricula"),
    ("Sou calouro e ninguém me explicou como escolher as disciplinas.", "matricula"),

    ("Uma empresa me chamou para trabalhar meio período enquanto estudo, preciso de assinatura da escola.", "estagio"),
    ("Quantas horas na empresa são exigidas para concluir o curso?", "estagio"),
    ("Meu supervisor na firma precisa validar o documento das minhas atividades.", "estagio"),
    ("A empresa quer prorrogar meu contrato por mais um semestre.", "estagio"),
    ("Onde acho o modelo do plano de atividades para o local onde vou trabalhar?", "estagio"),

    ("Terminei todas as cadeiras, quando posso receber meu canudo?", "colacao"),
    ("Quantas pessoas posso levar para a cerimônia?", "colacao"),
    ("Meu nome está escrito errado na lista dos formandos.", "colacao"),
    ("Passei num concurso e preciso do documento de conclusão antes da data oficial.", "colacao"),
    ("A solenidade vai acontecer no auditório ou de forma remota?", "colacao"),

    ("Acho que houve erro na soma dos meus pontos, como peço reavaliação?", "recurso"),
    ("Fui reprovado por faltas, mas apresentei atestado no período.", "recurso"),
    ("Quero questionar formalmente a decisão que indeferiu meu pedido.", "recurso"),
    ("O professor não registrou a nota do meu seminário, o que faço?", "recurso"),
    ("Perdi a avaliação porque estava doente, tenho direito a outra oportunidade?", "recurso"),

    ("A rede sem fio do prédio está caindo toda hora.", "outro"),
    ("Preciso de um papel que comprove que estudo aqui para o transporte.", "outro"),
    ("Vai ter expediente na secretaria durante o recesso?", "outro"),
    ("Quero conversar com alguém da coordenação sobre um assunto pessoal.", "outro"),
    ("Onde consulto as datas de início e fim do semestre?", "outro"),
]

# Mesmas 25 intenções do TESTE_FORMAL, escritas como aluno escreve no WhatsApp
TESTE_REAL = [
    ("professor, travei na hora de pegar as materia, o site fica dando erro", "matricula"),
    ("quero dar um tempo nos estudos esse semestre e voltar ano que vem", "matricula"),
    ("peguei uma cadeira errada, tem como tirar ela do meu horario?", "matricula"),
    ("sou novato aqui, ngm me falou como escolher as materia", "matricula"),
    ("as vaga de estrutura acabaram, sobra alguma pra mim?", "matricula"),

    ("arrumei um trampo numa empresa de TI, preciso que a escola assine uns papel", "estagio"),
    ("quantas hora tenho q cumprir na firma pra fechar o curso?", "estagio"),
    ("meu chefe pediu pra escola mandar uma carta apresentando eu", "estagio"),
    ("vou continuar mais 6 meses na empresa, precisa renovar alguma coisa?", "estagio"),
    ("posso usar meu emprego atual pra contar as hora obrigatoria?", "estagio"),

    ("acabei todas as materia, quando eu pego o canudo?", "colacao"),
    ("quantos parente eu posso levar no dia da formatura?", "colacao"),
    ("escreveram meu nome errado na lista dos que vao se formar", "colacao"),
    ("passei num concurso e preciso do papel de conclusao antes do prazo normal", "colacao"),
    ("vai ter ensaio antes do dia da cerimonia?", "colacao"),

    ("acho q o prof errou a soma da minha nota, da pra pedir pra ver de novo?", "recurso"),
    ("faltei a prova pq tava internado, tenho direito de fazer outra?", "recurso"),
    ("quero questionar a decisao que negou meu pedido", "recurso"),
    ("minhas falta tao erradas no sistema, como arrumo isso?", "recurso"),
    ("o prof nao colocou a nota do meu seminario ate agora", "recurso"),

    ("o wifi do predio ta caindo direto", "outro"),
    ("preciso de um papel comprovando que estudo ai pra pegar meia no onibus", "outro"),
    ("a secretaria abre no recesso?", "outro"),
    ("queria marcar uma conversa com a coordenacao", "outro"),
    ("onde vejo quando começa e termina o semestre?", "outro"),
]

In [4]:
treino = pd.DataFrame(TREINO, columns=["texto", "categoria"])
formal = pd.DataFrame(TESTE_FORMAL, columns=["texto", "categoria"])
real   = pd.DataFrame(TESTE_REAL, columns=["texto", "categoria"])

print(f"treino: {len(treino)} | teste formal: {len(formal)} | teste real: {len(real)}")
display(treino.categoria.value_counts().to_frame("exemplos").T)
treino.sample(5, random_state=0)

treino: 100 | teste formal: 25 | teste real: 25


categoria,matricula,estagio,colacao,recurso,outro
exemplos,20,20,20,20,20


,texto,categoria
26,Quem é o professor orientador de estágio do cu...,estagio
86,Onde encontro o calendário acadêmico atualizado?,outro
2,"Perdi o prazo de matrícula, ainda dá tempo de ...",matricula
55,A cerimônia de formatura será presencial ou on...,colacao
75,Meu pedido de dispensa de disciplina foi negad...,recurso


In [5]:
# A mesma intenção nos dois conjuntos de teste, lado a lado
pd.DataFrame({"formal": formal.texto, "real": real.texto, "categoria": formal.categoria}).head(6)

,formal,real,categoria
0,Não consegui garantir vaga nas cadeiras que es...,"professor, travei na hora de pegar as materia,...",matricula
1,Como faço pra sair de uma disciplina que pegue...,quero dar um tempo nos estudos esse semestre e...,matricula
2,Quero suspender meus estudos por um semestre e...,"peguei uma cadeira errada, tem como tirar ela ...",matricula
3,O portal não aceita minha inscrição nas turmas...,"sou novato aqui, ngm me falou como escolher as...",matricula
4,Sou calouro e ninguém me explicou como escolhe...,"as vaga de estrutura acabaram, sobra alguma pr...",matricula
5,Uma empresa me chamou para trabalhar meio perí...,"arrumei um trampo numa empresa de TI, preciso ...",estagio


---

## 1. Geração 1: Era estatística: TF-IDF + Regressão Logística

A receita clássica dos slides: **bag of words** com peso TF-IDF, e um classificador supervisionado por cima. O texto vira uma matriz de contagens ponderadas, cada palavra do vocabulário é uma coluna, e nada mais.

`ngram_range=(1, 2)` acrescenta pares de palavras ("colação de grau" vira um atributo próprio),é a versão mais barata dos *hand-crafted features* que apareceram na aula.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

inicio = time.time()
g1 = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=1000))
g1.fit(treino.texto, treino.categoria)
tempo_g1 = time.time() - inicio

acc_formal = g1.score(formal.texto, formal.categoria)
acc_real   = g1.score(real.texto, real.categoria)

RESULTADOS["1. TF-IDF + LR"] = dict(formal=acc_formal, real=acc_real, rotulados=100, seg=tempo_g1)
print(f"acurácia (formal): {acc_formal:.0%}")
print(f"acurácia (real):   {acc_real:.0%}")
print(f"treino em {tempo_g1:.2f}s, vocabulário de {len(g1[0].vocabulary_)} atributos")

acurácia (formal): 92%
acurácia (real):   80%
treino em 0.16s, vocabulário de 1043 atributos


### Onde ele erra  e por quê?

Rode a célula abaixo e olhe as mensagens que ele errou no conjunto **real**.

In [7]:
def erros(modelo, df, encode=None):
    X = encode(df.texto.tolist()) if encode else df.texto
    pred = modelo.predict(X)
    err = df.assign(previsto=pred).query("categoria != previsto")
    return err[["categoria", "previsto", "texto"]]

erros(g1, real)

,categoria,previsto,texto
0,matricula,outro,"professor, travei na hora de pegar as materia,..."
2,matricula,outro,"peguei uma cadeira errada, tem como tirar ela ..."
4,matricula,estagio,"as vaga de estrutura acabaram, sobra alguma pr..."
13,colacao,estagio,passei num concurso e preciso do papel de conc...
14,colacao,outro,vai ter ensaio antes do dia da cerimonia?


### O limite estrutural do TF-IDF

> *"médico" e "doutor" são vetores ortogonais*. Vamos medir isso.

Para o TF-IDF, duas frases só são parecidas se **repetirem as mesmas palavras**. Não existe
noção de significado — existe sobreposição de string.

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

pares = [
    ("matrícula", "inscrição"),
    ("prova", "avaliação"),
    ("Perdi a prova por motivo de doença.", "Faltei ao exame porque estava internado."),
    ("O wifi está fora do ar.",             "A rede sem fio caiu."),
    ("Preciso do meu diploma.",             "Preciso do meu canudo."),   # <- olhe este
]

vec = g1[0]   # o TfidfVectorizer já treinado
for a, b in pares:
    s = cosine_similarity(vec.transform([a]), vec.transform([b]))[0, 0]
    print(f"{s:.2f}   {a!r}  x  {b!r}")

0.00   'matrícula'  x  'inscrição'
0.00   'prova'  x  'avaliação'
0.00   'Perdi a prova por motivo de doença.'  x  'Faltei ao exame porque estava internado.'
0.00   'O wifi está fora do ar.'  x  'A rede sem fio caiu.'
0.77   'Preciso do meu diploma.'  x  'Preciso do meu canudo.'


Leia com atenção, porque tem duas coisas acontecendo:

- **Similaridade 0.00** para pares que significam a mesma coisa. E não é porque as palavras estão   faltando: "matrícula" e "inscrição" **estão as duas** no vocabulário de treino. Elas são   simplesmente colunas diferentes da matriz, ortogonais, sem nenhuma relação. É o *"médico e  doutor são vetores ortogonais"* do plano de aula, medido.
- O último par tem a **maior** similaridade da lista. Não por significado: "diploma" e "canudo"
  não têm relação nenhuma para o modelo. É o `"Preciso do meu"` repetido nas duas frases. O
  TF-IDF está medindo sobreposição de string e chamando isso de semelhança.

Guarde esses números: vamos repetir exatamente este teste na Geração 2.

---

## 2. Geração 2: Era neural pré-Transformer: embeddings

A ideia do Word2Vec/GloVe: **significado é posição no espaço vetorial**. Em vez de uma coluna por palavra, cada frase vira um vetor denso de algumas centenas de dimensões, onde frases próximas em sentido ficam próximas em distância.

Vamos usar um modelo multilíngue do `sentence-transformers`, que já vem treinado, não precisamos de corpus nem de GPU. **O classificador é o mesmo da Geração 1**: só trocamos a representação da entrada.

> Rigor histórico: este modelo por dentro já é um Transformer. O que estamos demonstrando aqui é o *salto de representação* (esparsa → densa), que é o ganho da era 2013–2017. Treinar um Word2Vec do zero não caberia em 12 minutos.


In [9]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

inicio = time.time()
Xtreino = encoder.encode(treino.texto.tolist(), show_progress_bar=False)
g2 = LogisticRegression(max_iter=1000).fit(Xtreino, treino.categoria)
tempo_g2 = time.time() - inicio

acc_formal = g2.score(encoder.encode(formal.texto.tolist()), formal.categoria)
acc_real   = g2.score(encoder.encode(real.texto.tolist()), real.categoria)

RESULTADOS["2. Embeddings + LR"] = dict(formal=acc_formal, real=acc_real, rotulados=100, seg=tempo_g2)
print(f"cada frase virou um vetor de {Xtreino.shape[1]} dimensões (contra {len(g1[0].vocabulary_)} do TF-IDF)")
print(f"acurácia (formal): {acc_formal:.0%}")
print(f"acurácia (real):   {acc_real:.0%}")
print(f"treino em {tempo_g2:.2f}s")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

cada frase virou um vetor de 384 dimensões (contra 1043 do TF-IDF)
acurácia (formal): 92%
acurácia (real):   72%
treino em 2.92s


In [10]:
# O MESMO teste de similaridade da Geração 1, agora com embeddings
for a, b in pares:
    s = cosine_similarity(encoder.encode([a]), encoder.encode([b]))[0, 0]
    print(f"{s:.2f}   {a!r}  x  {b!r}")

0.77   'matrícula'  x  'inscrição'
0.79   'prova'  x  'avaliação'
0.73   'Perdi a prova por motivo de doença.'  x  'Faltei ao exame porque estava internado.'
0.65   'O wifi está fora do ar.'  x  'A rede sem fio caiu.'
0.47   'Preciso do meu diploma.'  x  'Preciso do meu canudo.'


Os quatro primeiros pares saíram de 0.00 para valores altos: os sinônimos agora **estão perto** no espaço vetorial. E o último par, o que o TF-IDF achou que era o mais parecido de todos, caiu, porque `"Preciso do meu"` não carrega significado nenhum.

É essa a mudança da era 2013–2017: **significado virou posição no espaço**. Sem essa mudança não existe busca semântica, e sem busca semântica não existe RAG.

In [11]:
erros(g2, real, encode=lambda t: encoder.encode(t))

,categoria,previsto,texto
0,matricula,outro,"professor, travei na hora de pegar as materia,..."
3,matricula,colacao,"sou novato aqui, ngm me falou como escolher as..."
4,matricula,colacao,"as vaga de estrutura acabaram, sobra alguma pr..."
5,estagio,recurso,"arrumei um trampo numa empresa de TI, preciso ..."
7,estagio,recurso,meu chefe pediu pra escola mandar uma carta ap...
21,outro,recurso,preciso de um papel comprovando que estudo ai ...
22,outro,estagio,a secretaria abre no recesso?


### Provavelmente a acurácia não subiu. E isso é o ponto.

Se o seu palpite era "a Geração 2 ganha", você está em boa companhia, e o número provavelmente desmentiu. Nos testes que fizemos ao preparar esta aula, o TF-IDF ficou **igual ou melhor** que os embeddings no conjunto real.

Três razões, todas honestas:

1. **100 exemplos é muito pouco** para uma Regressão Logística aprender a separar 384 dimensões    densas. O TF-IDF entrega ao classificador atributos já quase prontos ("colação", "estágio"),    enquanto o embedding entrega uma posição no espaço que o classificador ainda precisa aprender  a fatiar.
2. **O ruído de 25 exemplos** (lembra? 1 erro = 4 pontos) engole qualquer diferença pequena.
3. Nossas mensagens informais **ainda compartilham bastante vocabulário** com o treino. O TF-IDF não precisou entender nada, bastou reconhecer palavras.

**Então o que a Geração 2 trouxe de verdade?** Compare as duas células de similaridade: o TF-IDF dá ~0.0 para frases sinônimas; o embedding dá um valor alto. Esse ganho é **da representação**, é determinístico, e não depende de sorte no conjunto de teste. É ele, não a acurácia deste recorte, que sustenta a busca semântica e todo o RAG.

> Lição de método: **um benchmark pequeno não decide nada.** Quem te mostrar uma tabela de
> acurácia sem dizer o tamanho do conjunto de teste está vendendo alguma coisa.


## 3. Geração 3: Era Transformer: LLM em zero-shot

Aqui a mudança não é de algoritmo, é de **regime de trabalho**: não existe treino, não existe conjunto rotulado. Existe **um prompt**.

### Passo 1: a credencial (gratuita)

Escolha **um** dos dois provedores. Os dois são gratuitos e os dois já estão na lista de
pré-requisitos da disciplina.

**Opção A - Hugging Face** *(recomendada: você vai precisar dessa conta o curso inteiro)*

1. Abra [este link](https://huggingface.co/settings/tokens/new?ownUserPermissions=inference.serverless.write&tokenType=fineGrained)
   — ele já vem com o tipo **Fine-grained** e a permissão **`Make calls to Inference Providers`**
   marcados. É essa permissão que autoriza a chamada ao LLM; sem ela, a resposta é `401`.
2. Dê um nome (ex.: `colab-teia-llm`) → **Create token** → **copie agora**: o valor só aparece
   uma vez.
3. No Colab, clique na **chave 🔑** na barra lateral esquerda (*Secrets*).
4. `+ Adicionar novo secret` → nome exatamente **`HF_TOKEN`** → cole o token.
5. Ligue o botão **Acesso ao notebook**. *(É o passo que todo mundo esquece.)*

**Opção B - Google AI Studio**

1. <https://aistudio.google.com/apikey> → gere uma chave (login com conta Google).
2. Mesmos passos 3 a 5 acima, com o nome **`GOOGLE_API_KEY`**.

> **Nunca** cole a credencial direto numa célula de código. Notebook vai para o GitHub; chave vazada é chave cancelada — o HF varre repositórios públicos e revoga sozinho.

> As duas camadas gratuitas têm limite (o HF dá um crédito mensal; o Google, uma cota por minuto). Nosso laboratório faz **2 chamadas** no total, então cabe folgado em qualquer uma das duas. Se uma cair no meio da aula, troque o valor de `PROVIDER` na célula abaixo e siga.

In [31]:
PROVIDER = "gemini"      # ou "gemini" — troque aqui se o outro estiver fora do ar

def segredo(nome):
    try:
        from google.colab import userdata
        return userdata.get(nome)
    except Exception:                    # fora do Colab, digita na hora
        import getpass
        return getpass.getpass(f"Cole seu {nome}: ")


if PROVIDER == "huggingface":
    from huggingface_hub import InferenceClient
    MODEL = "meta-llama/Llama-3.3-70B-Instruct"          # catálogo muda: confira em huggingface.co/models
    cliente = InferenceClient(api_key=segredo("HF_TOKEN"))

    def chamar_llm(prompt: str) -> str:
        """Chamada mínima a um LLM. Vira perguntar() multi-backend na E04."""
        r = cliente.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1000,                     # sem isso a resposta corta no meio do JSON
        )
        return r.choices[0].message.content

elif PROVIDER == "gemini":
    from google import genai
    MODEL = "gemini-2.5-flash"                  # catálogo muda: confira em ai.google.dev/models
    cliente = genai.Client(api_key=segredo("GOOGLE_API_KEY"))

    def chamar_llm(prompt: str) -> str:
        """Chamada mínima a um LLM. Vira perguntar() multi-backend na E04."""
        return cliente.models.generate_content(model=MODEL, contents=prompt).text


try:
    print(f"[{PROVIDER} · {MODEL}]", chamar_llm("Responda em uma frase: o que é NLP?")) # Testando o acesso
except Exception as e:
    print("A chamada falhou:", type(e).__name__)
    print("  429 -> cota ou crédito esgotado; troque o PROVIDER ou gere credencial nova")
    print("  401 / 403 -> credencial inválida, ou o secret está sem 'Acesso ao notebook'")
    raise

[gemini · gemini-2.5-flash] PNL (Processamento de Linguagem Natural) é um ramo da inteligência artificial que permite aos computadores compreender, interpretar e gerar a linguagem humana.


### Passo 2: o prompt

Repare no que **não** existe nesta célula: nenhum exemplo rotulado, nenhum treino, nenhum
`.fit()`. A descrição da tarefa em português é o modelo inteiro.

Classificamos as 25 mensagens **numa única chamada** — mais rápido e bem mais leve para a cota
gratuita do que 25 chamadas separadas.

In [30]:
CATEGORIAS = ["matricula", "estagio", "colacao", "recurso", "outro"]

def classificar_com_llm(textos):
    numeradas = "\n".join(f"{i+1}. {t}" for i, t in enumerate(textos))
    prompt = f"""Você faz a triagem das mensagens que chegam à coordenação de um curso superior.

Classifique CADA mensagem em exatamente uma destas categorias:
{", ".join(CATEGORIAS)}

Responda SOMENTE com um array JSON de strings, na mesma ordem e com o mesmo número de itens
das mensagens. Sem explicação, sem markdown.

Mensagens:
{numeradas}"""

    resposta = chamar_llm(prompt)
    pred = json.loads(re.search(r"\[.*\]", resposta, re.S).group())
    assert len(pred) == len(textos), f"esperava {len(textos)} rótulos, vieram {len(pred)}"
    # o modelo às vezes devolve "Matrícula" em vez de "matricula" — normalizamos antes de comparar
    limpar = lambda s: unicodedata.normalize("NFKD", s.strip().lower()).encode("ascii", "ignore").decode()
    return [limpar(p) for p in pred]


inicio = time.time()
pred_formal = classificar_com_llm(formal.texto.tolist())
pred_real   = classificar_com_llm(real.texto.tolist())
tempo_g3 = time.time() - inicio

acc_formal = (pd.Series(pred_formal).values == formal.categoria.values).mean()
acc_real   = (pd.Series(pred_real).values == real.categoria.values).mean()

RESULTADOS["3. LLM zero-shot"] = dict(formal=acc_formal, real=acc_real, rotulados=0, seg=tempo_g3)
print(f"acurácia (formal): {acc_formal:.0%}")
print(f"acurácia (real):   {acc_real:.0%}")
print(f"{tempo_g3:.1f}s para as duas chamadas — e zero exemplo rotulado")

acurácia (formal): 100%
acurácia (real):   100%
14.7s para as duas chamadas — e zero exemplo rotulado


In [29]:
# Onde o LLM errou no conjunto real
real.assign(previsto=pred_real).query("categoria != previsto")[["categoria", "previsto", "texto"]]

,categoria,previsto,texto


> **Duas ressalvas honestas**, para ninguém sair daqui com fé demais:
>
> - O resultado **não é determinístico**. Rode de novo e a acurácia pode mudar. Controlar isso (`temperature`, saída estruturada).
> - O LLM **erra com confiança absoluta**. Ele nunca devolve "não sei" devolve uma categoria errada com a mesma cara de quem acertou.


## 4. Comparação

In [26]:
tabela = pd.DataFrame(RESULTADOS).T
tabela.columns = ["Acurácia (formal)", "Acurácia (real)", "Exemplos rotulados", "Tempo (s)"]
tabela["Queda formal → real"] = tabela["Acurácia (formal)"] - tabela["Acurácia (real)"]
tabela.style.format({
    "Acurácia (formal)": "{:.0%}", "Acurácia (real)": "{:.0%}",
    "Queda formal → real": "{:+.0%}", "Exemplos rotulados": "{:.0f}", "Tempo (s)": "{:.1f}",
})

,Acurácia (formal),Acurácia (real),Exemplos rotulados,Tempo (s),Queda formal → real
1. TF-IDF + LR,92%,80%,100,0.2,+12%
2. Embeddings + LR,92%,72%,100,2.9,+20%
3. LLM zero-shot,84%,80%,0,3.1,+4%


As outras duas linhas têm rotulados=100 porque as Gerações 1 e 2 chamam `.fit(treino.texto, treino.categoria)`, elas precisam das 100 mensagens já rotuladas para existir. Tire o treino e não sobra classificador nenhum.

A Geração 3 nunca chama `.fit()`. Não há treino, não há treino.categoria em lugar nenhum do fluxo. O que define a tarefa é o texto do prompt: os cinco nomes de categoria e a instrução em português. Por isso zero é literalmente a contagem de exemplos rotulados que o modelo viu para aprender a tarefa.

A distinção fina, que provavelmente vai aparecer na aula: os rótulos do teste (formal.categoria, real.categoria) são usados mas só depois, para conferir o gabarito e calcular a acurácia. Isso é medir, não treinar. Os três modelos usam esses mesmos 25 rótulos de teste em condições idênticas; o que difere é que dois deles consumiram mais 100 antes de começar

### Leia a tabela antes de responder

Quatro perguntas para orientar a leitura:

1. As acurácias ficaram próximas? Com 25 exemplos de teste, quantos **itens** de diferença    existem de fato entre a maior e a menor? Isso é evidência de quê?
2. Se a acurácia **não** separa bem as três gerações, qual coluna separa?
3. A Geração 3 usou **zero** exemplo rotulado. Quanto custaram, em trabalho humano, os 100 exemplos que as Gerações 1 e 2 exigiram, e quem os rotularia numa coordenação de verdade?
4. O tempo medido é o de **treino**. Falta na tabela o tempo de **cada previsão**. Qual das três você imagina que é a mais lenta por mensagem? E qual funciona sem internet?

### A pergunta da aula

**Em que situação você ainda escolheria a Geração 1?**

A resposta não é "nunca". Pense em: latência por mensagem, custo por 1000 chamadas, previsibilidade da saída, funcionamento sem internet, e dado sensível que não pode sair da instituição.

Responda em **um parágrafo** na célula abaixo.

**Sua resposta:**

_(escreva aqui)_

---

### Resumo

- NLP não nasceu em 2017. O que mudou foi *como* se resolve, não *o que* se resolve.
- TF-IDF não é descartável: com vocabulário estável ele é rápido, barato, previsível e, num conjunto   pequeno como o de hoje, competitivo com coisa muito mais moderna.
- O ganho dos embeddings é de **representação** (similaridade semântica), e nem sempre aparece na acurácia. .
- Tabela de acurácia sem tamanho de conjunto de teste não é evidência.
- O LLM eliminou o conjunto rotulado. Em troca, trouxe latência, custo por chamada, não determinismo e alucinação.

### Referências da aula

| Recurso | Link |
|---|---|
| HF LLM Course — Cap. 1 | https://huggingface.co/learn/llm-course/chapter1/1 |
| *The Illustrated Word2Vec* | https://jalammar.github.io/illustrated-word2vec/ |
| scikit-learn — TF-IDF | https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction |
| Sentence Transformers | https://sbert.net/ |
| Vaswani et al., 2017 — *Attention Is All You Need* | https://arxiv.org/abs/1706.03762 |